## 各个SV工具输出的vcf文件格式差异很大，不利于统计，为了便于后续分析，计划在不改动原始vcf结果数据的前提下对部分BND添加注释信息，且在后续统计时优先阅读。

## （此pipeline紧接sv工具结果）

# 一、需要用到的脚本

### annotate_bnd_with_inferred_svtype.py：

In [ ]:
#!/usr/bin/env python3
import argparse
import csv
import gzip
import re
from collections import Counter, defaultdict
from pathlib import Path


DEFAULT_RESULT_ROOT = "/mnt/home/ygjx/chenkejin/share_group_folder_ygjx/Pancreatic_datasets/PDAC_WGS/sv_tools_results"
DEFAULT_PAIR_LIST = "/mnt/home/ygjx/chenkejin/80_upset/80_pairs.normal_tumor.tsv"
DEFAULT_OUTDIR = "/mnt/home/ygjx/chenkejin/raw_sv_vcf_audit"

TOOLS = ["cue", "lumpy", "gridss", "manta", "delly", "svaba"]

IMPORTANT_INFO_KEYS = [
    "SVTYPE", "END", "SVLEN", "CHR2", "END2", "POS2", "MATEID", "EVENT",
    "STRANDS", "CT", "CIPOS", "CIEND", "PRECISE", "IMPRECISE",
    "HOMLEN", "HOMSEQ", "CIPOS95", "CIEND95",
    "SU", "PE", "PR", "SR", "RP", "AS", "QUAL", "SRQ", "MAPQ",
    "BND_DEPTH", "REF", "REFPAIR", "PAIR_COUNT",
]

RAW_EXCLUDE_PATTERNS = [
    ".tbi",
    ".csi",
    ".idx",
    "bnd_reclass",
    "reclass",
    "normalized",
    "merged_sv_events",
]


def parse_args():
    parser = argparse.ArgumentParser(
        description=(
            "Audit raw SV VCFs from six callers for 80 PDAC tumor-normal pairs. "
            "This script does not modify VCFs. It summarizes whether BND records "
            "can be used directly or need caller-specific inference."
        )
    )
    parser.add_argument("--result-root", default=DEFAULT_RESULT_ROOT)
    parser.add_argument("--pair-list", default=DEFAULT_PAIR_LIST)
    parser.add_argument("--outdir", default=DEFAULT_OUTDIR)
    parser.add_argument("--tools", default=",".join(TOOLS), help="Comma-separated tool list.")
    parser.add_argument(
        "--max-records-per-vcf",
        type=int,
        default=0,
        help="0 means parse all records; otherwise parse only the first N records per VCF.",
    )
    parser.add_argument(
        "--include-nonpass",
        action="store_true",
        help="Only affects recommendations. Raw counts for all and PASS are always reported.",
    )
    parser.add_argument(
        "--allow-pon-removed",
        action="store_true",
        help="By default gridss/gripss pon_removed files are down-ranked. This option removes that penalty.",
    )
    return parser.parse_args()


def clean_chr(chrom):
    x = str(chrom or "").strip()
    x = re.sub(r"^chr", "", x, flags=re.I)
    return x


def sample_core(sample):
    s = str(sample)
    return re.sub(r"[TNP]$", "", s)


def open_text(path):
    p = str(path)
    if p.endswith(".gz"):
        return gzip.open(p, "rt", errors="replace")
    return open(p, "rt", errors="replace")


def parse_info(info_text):
    out = {}
    if not info_text or info_text == ".":
        return out
    for item in info_text.split(";"):
        if not item:
            continue
        if "=" in item:
            k, v = item.split("=", 1)
            out[k] = v
        else:
            out[item] = True
    return out


def norm_svtype(value):
    sv = str(value or "").strip().upper()
    if sv in {"", ".", "NA", "NAN", "NONE"}:
        return "UNKNOWN"
    if sv in {"DELETION"}:
        return "DEL"
    if sv in {"DUPLICATION", "IDUP", "DUP:TANDEM", "TANDEM_DUPLICATION"}:
        return "DUP"
    if sv in {"INVERSION"}:
        return "INV"
    if sv in {"INSERTION"}:
        return "INS"
    if sv in {"TRANSLOCATION", "CTX"}:
        return "TRA"
    if sv in {"BREAKEND"}:
        return "BND"
    if sv in {"SINGLE_BREAKEND", "SINGLE"}:
        return "SGL"
    return sv


def infer_svtype_from_alt(alt):
    alt = str(alt or "")
    m = re.search(r"<([^>]+)>", alt)
    if m:
        return norm_svtype(m.group(1))
    if "[" in alt or "]" in alt:
        return "BND"
    return "UNKNOWN"


def parse_bnd_alt_mate(alt):
    alt = str(alt or "")
    m = re.search(r"[\[\]]([^:\[\]]+):([0-9]+)[\[\]]", alt)
    if not m:
        return "", ""
    return clean_chr(m.group(1)), m.group(2)


def is_pass_filter(flt):
    return flt in {"PASS", "."}


def split_items(value):
    if value is None:
        return []
    return [x for x in re.split(r"[,|]", str(value)) if x not in {"", "."}]


def read_pairs(pair_list):
    path = Path(pair_list)
    if not path.exists():
        raise SystemExit(f"ERROR: pair list not found: {pair_list}")

    with open(path, "rt", errors="replace") as handle:
        first = handle.readline().rstrip("\n")
        if not first:
            return []
        fields = first.split("\t")
        has_header = any(x.lower() in {"pair_id", "tumor_id", "normal_id", "tumor", "normal"} for x in fields)
        handle.seek(0)

        pairs = []
        if has_header:
            reader = csv.DictReader(handle, delimiter="\t")
            for row in reader:
                tumor = row.get("tumor_id") or row.get("tumor") or row.get("tumour_id") or row.get("tumour")
                normal = row.get("normal_id") or row.get("normal")
                pair_id = row.get("pair_id") or row.get("pair")
                if not pair_id and tumor and normal:
                    pair_id = f"{tumor}_vs_{normal}"
                if not tumor and pair_id and "_vs_" in pair_id:
                    tumor, normal = pair_id.split("_vs_", 1)
                if pair_id and tumor and normal:
                    pairs.append({"pair_id": pair_id, "tumor": tumor, "normal": normal, "core": sample_core(tumor)})
        else:
            reader = csv.reader(handle, delimiter="\t")
            for row in reader:
                if len(row) >= 2:
                    tumor, normal = row[0], row[1]
                    pairs.append({"pair_id": f"{tumor}_vs_{normal}", "tumor": tumor, "normal": normal, "core": sample_core(tumor)})
        return pairs


def discover_vcfs(result_root, tools):
    root = Path(result_root)
    if not root.exists():
        raise SystemExit(f"ERROR: result root not found: {result_root}")

    candidates = []
    for p in root.rglob("*"):
        if not p.is_file():
            continue
        name_lower = p.name.lower()
        full_lower = str(p).lower()
        if not (name_lower.endswith(".vcf") or name_lower.endswith(".vcf.gz")):
            continue
        if any(x in full_lower for x in RAW_EXCLUDE_PATTERNS):
            continue

        detected_tool = ""
        for tool in tools:
            if re.search(rf"(^|[/_.-]){re.escape(tool)}($|[/_.-])", full_lower):
                detected_tool = tool
                break
        if not detected_tool:
            # gripss output belongs to gridss workflow.
            if "gripss" in full_lower or "gridss" in full_lower:
                detected_tool = "gridss"
            elif "svaba" in full_lower:
                detected_tool = "svaba"
            elif "delly" in full_lower:
                detected_tool = "delly"
            elif "lumpy" in full_lower:
                detected_tool = "lumpy"
            elif "manta" in full_lower:
                detected_tool = "manta"
            elif "cue" in full_lower:
                detected_tool = "cue"

        if detected_tool in tools:
            candidates.append({"tool": detected_tool, "path": str(p)})
    return candidates


def score_candidate_for_pair(candidate, pair, allow_pon_removed=False):
    path = candidate["path"]
    tool = candidate["tool"]
    low = path.lower()
    name = Path(path).name
    name_low = name.lower()
    tumor = pair["tumor"]
    normal = pair["normal"]
    pair_id = pair["pair_id"]
    core = pair["core"]

    score = 0
    reasons = []

    if pair_id.lower() in low:
        score += 140
        reasons.append("pair_id")
    if tumor.lower() in low:
        score += 60
        reasons.append("tumor")
    if normal.lower() in low:
        score += 45
        reasons.append("normal")
    if core and core.lower() in low:
        score += 30
        reasons.append("core")
    if "somatic" in low:
        score += 40
        reasons.append("somatic")
    if tool in low:
        score += 10
        reasons.append("tool")

    # Caller-specific preferences and traps.
    if tool == "cue":
        if "_raw" in name_low or name_low.endswith("raw.vcf") or name_low.endswith("raw.vcf.gz"):
            score -= 160
            reasons.append("penalty_raw_single_sample")
        if "_somatic" in name_low or "somatic" in name_low:
            score += 60
            reasons.append("cue_somatic")
    elif tool == "gridss":
        if "pon_filtered" in low:
            score += 80
            reasons.append("gridss_pon_filtered")
        if "gripss.filtered" in low or "gripss_filter" in low:
            score += 40
            reasons.append("gripss_filtered")
        if "pon_removed" in low and not allow_pon_removed:
            score -= 180
            reasons.append("penalty_pon_removed")
    elif tool == "svaba":
        if ".somatic.sv" in name_low:
            score += 90
            reasons.append("svaba_somatic_sv")
        if ".somatic.indel" in name_low or "indel" in name_low:
            score -= 140
            reasons.append("penalty_svaba_indel")
    elif tool == "manta":
        if "somatic" in name_low or "somaticsv" in name_low:
            score += 60
            reasons.append("manta_somatic")
        if "diploid" in name_low:
            score -= 40
            reasons.append("penalty_diploid")
    elif tool == "delly":
        if "somatic" in name_low:
            score += 70
            reasons.append("delly_somatic")
    elif tool == "lumpy":
        if "somatic" in name_low:
            score += 60
            reasons.append("lumpy_somatic")

    return score, ";".join(reasons)


def select_vcfs(candidates, pairs, tools, allow_pon_removed=False):
    by_tool = defaultdict(list)
    for c in candidates:
        by_tool[c["tool"]].append(c)

    selected = []
    for pair in pairs:
        for tool in tools:
            best = None
            best_score = -10**9
            best_reasons = ""
            for cand in by_tool.get(tool, []):
                score, reasons = score_candidate_for_pair(cand, pair, allow_pon_removed)
                if score > best_score:
                    best = cand
                    best_score = score
                    best_reasons = reasons
            if best and best_score > 0:
                selected.append({
                    "pair_id": pair["pair_id"],
                    "tumor": pair["tumor"],
                    "normal": pair["normal"],
                    "tool": tool,
                    "vcf": best["path"],
                    "match_score": best_score,
                    "match_reasons": best_reasons,
                    "match_status": "OK",
                })
            else:
                selected.append({
                    "pair_id": pair["pair_id"],
                    "tumor": pair["tumor"],
                    "normal": pair["normal"],
                    "tool": tool,
                    "vcf": "NA",
                    "match_score": best_score if best else "NA",
                    "match_reasons": best_reasons,
                    "match_status": "MISSING",
                })
    return selected


def parse_vcf(vcf_path, max_records=0):
    info_header = set()
    format_header = set()
    filter_header = set()
    sample_names = []

    total = 0
    pass_n = 0
    filter_counts = Counter()
    svtype_counts_all = Counter()
    svtype_counts_pass = Counter()
    alt_class_counts = Counter()
    info_presence = Counter()
    format_presence = Counter()
    bnd = Counter()
    bnd_infer = Counter()
    first_records = []

    with open_text(vcf_path) as handle:
        for line in handle:
            line = line.rstrip("\n")
            if not line:
                continue
            if line.startswith("##INFO="):
                m = re.search(r"ID=([^,>]+)", line)
                if m:
                    info_header.add(m.group(1))
                continue
            if line.startswith("##FORMAT="):
                m = re.search(r"ID=([^,>]+)", line)
                if m:
                    format_header.add(m.group(1))
                continue
            if line.startswith("##FILTER="):
                m = re.search(r"ID=([^,>]+)", line)
                if m:
                    filter_header.add(m.group(1))
                continue
            if line.startswith("#CHROM"):
                parts = line.split("\t")
                sample_names = parts[9:] if len(parts) > 9 else []
                continue
            if line.startswith("#"):
                continue

            parts = line.split("\t")
            if len(parts) < 8:
                continue
            chrom, pos, vid, ref, alt, qual, flt, info_text = parts[:8]
            fmt = parts[8] if len(parts) > 8 else ""
            info = parse_info(info_text)

            total += 1
            is_pass = is_pass_filter(flt)
            if is_pass:
                pass_n += 1

            filter_counts[flt] += 1
            svtype = norm_svtype(info.get("SVTYPE") or infer_svtype_from_alt(alt))
            svtype_counts_all[svtype] += 1
            if is_pass:
                svtype_counts_pass[svtype] += 1

            if "[" in alt or "]" in alt:
                alt_class_counts["breakend_alt"] += 1
            elif re.search(r"<[^>]+>", alt):
                alt_class_counts["symbolic_alt"] += 1
            else:
                alt_class_counts["sequence_or_other_alt"] += 1

            for k in IMPORTANT_INFO_KEYS:
                if k in info:
                    info_presence[k] += 1
            if fmt:
                for k in fmt.split(":"):
                    if k:
                        format_presence[k] += 1

            if svtype == "BND" or "[" in alt or "]" in alt:
                bnd["bnd_total"] += 1
                if is_pass:
                    bnd["bnd_pass"] += 1
                mate_chr_alt, mate_pos_alt = parse_bnd_alt_mate(alt)
                chr2 = clean_chr(info.get("CHR2", ""))
                end = info.get("END") or info.get("END2") or info.get("POS2")
                strands = info.get("STRANDS")
                ct = info.get("CT")
                mateid = info.get("MATEID")

                if mate_chr_alt:
                    bnd["bnd_alt_mate_parseable"] += 1
                if mateid:
                    bnd["bnd_has_mateid"] += 1
                if chr2:
                    bnd["bnd_has_chr2"] += 1
                if end:
                    bnd["bnd_has_end_or_pos2"] += 1
                if strands:
                    bnd["bnd_has_strands"] += 1
                if ct:
                    bnd["bnd_has_ct"] += 1

                mate_chr = chr2 or mate_chr_alt
                if mate_chr:
                    if clean_chr(chrom) != clean_chr(mate_chr):
                        bnd["bnd_interchrom_mate"] += 1
                    else:
                        bnd["bnd_intrachrom_mate"] += 1

                if mate_chr and clean_chr(chrom) != clean_chr(mate_chr):
                    bnd_infer["bnd_inferable_TRA_by_mate_chr"] += 1
                elif mate_chr and (strands or ct):
                    bnd_infer["bnd_has_same_chr_orientation_info"] += 1
                elif mate_chr:
                    bnd_infer["bnd_has_mate_but_no_orientation"] += 1
                else:
                    bnd_infer["bnd_no_reliable_mate"] += 1

            if len(first_records) < 5:
                first_records.append({
                    "chrom": chrom,
                    "pos": pos,
                    "id": vid,
                    "alt": alt,
                    "filter": flt,
                    "svtype": svtype,
                    "info_keys": ",".join(sorted(info.keys())[:30]),
                    "format": fmt,
                })

            if max_records and total >= max_records:
                break

    return {
        "total_records": total,
        "pass_records": pass_n,
        "nonpass_records": total - pass_n,
        "sample_names": sample_names,
        "info_header": sorted(info_header),
        "format_header": sorted(format_header),
        "filter_header": sorted(filter_header),
        "filter_counts": filter_counts,
        "svtype_counts_all": svtype_counts_all,
        "svtype_counts_pass": svtype_counts_pass,
        "alt_class_counts": alt_class_counts,
        "info_presence": info_presence,
        "format_presence": format_presence,
        "bnd": bnd,
        "bnd_infer": bnd_infer,
        "first_records": first_records,
    }


def pct(n, d):
    if not d:
        return 0.0
    return n / d * 100.0


def make_recommendation(stats, include_nonpass=False):
    denom = stats["total_records"] if include_nonpass else stats["pass_records"]
    sv_counts = stats["svtype_counts_all"] if include_nonpass else stats["svtype_counts_pass"]
    bnd_n = sv_counts.get("BND", 0)
    if denom == 0:
        return "EMPTY_OR_NO_PASS", "No records available under selected scope."

    bnd_frac = bnd_n / denom
    bnd_total = stats["bnd"].get("bnd_total", 0)
    mate_parseable = stats["bnd"].get("bnd_alt_mate_parseable", 0) + stats["bnd"].get("bnd_has_chr2", 0)
    orientation = stats["bnd"].get("bnd_has_strands", 0) + stats["bnd"].get("bnd_has_ct", 0)
    interchrom = stats["bnd"].get("bnd_interchrom_mate", 0)

    non_bnd_types = [k for k, v in sv_counts.items() if k not in {"BND", "UNKNOWN"} and v > 0]

    if bnd_frac < 0.05 and non_bnd_types:
        return "DIRECT_USE_SVTYPE", "Most records already have explicit non-BND SVTYPE."
    if bnd_frac >= 0.05 and bnd_total > 0:
        mate_rate = mate_parseable / max(1, bnd_total)
        orient_rate = orientation / max(1, bnd_total)
        inter_rate = interchrom / max(1, bnd_total)
        if mate_rate >= 0.80 and orient_rate >= 0.50:
            return "NEED_CALLER_SPECIFIC_BND_INFERENCE", "BND has mate and orientation fields; infer DEL/DUP/INV/TRA only with caller-specific rules."
        if mate_rate >= 0.80 and inter_rate >= 0.50:
            return "CAN_INFER_TRA_BUT_NOT_ALL_TYPES", "Mate chromosome is usually available; TRA is inferable, but DEL/DUP/INV need orientation evidence."
        if mate_rate >= 0.50:
            return "PARTIAL_BND_INFERENCE_ONLY", "Some BND records have mate information; do not force all BND into canonical SV types."
        return "KEEP_BND_OR_EXCLUDE_FROM_TYPED_ANALYSIS", "BND lacks reliable mate/orientation fields for type conversion."
    if sv_counts.get("UNKNOWN", 0) / denom > 0.20:
        return "NEED_FORMAT_INSPECTION", "Many records lack interpretable SVTYPE."
    return "DIRECT_USE_SVTYPE", "SVTYPE is interpretable without BND conversion."


def write_tsv(path, rows, fieldnames):
    with open(path, "wt", newline="") as handle:
        writer = csv.DictWriter(handle, delimiter="\t", fieldnames=fieldnames, extrasaction="ignore")
        writer.writeheader()
        for row in rows:
            writer.writerow(row)


def main():
    args = parse_args()
    tools = [x.strip() for x in args.tools.split(",") if x.strip()]
    outdir = Path(args.outdir)
    outdir.mkdir(parents=True, exist_ok=True)

    pairs = read_pairs(args.pair_list)
    candidates = discover_vcfs(args.result_root, tools)
    selected = select_vcfs(candidates, pairs, tools, allow_pon_removed=args.allow_pon_removed)

    write_tsv(
        outdir / "raw_vcf_candidates.all.tsv",
        candidates,
        ["tool", "path"],
    )
    write_tsv(
        outdir / "raw_vcf_selected_manifest.tsv",
        selected,
        ["pair_id", "tumor", "normal", "tool", "match_status", "match_score", "match_reasons", "vcf"],
    )

    per_file = []
    svtype_rows = []
    filter_rows = []
    info_rows = []
    format_rows = []
    example_rows = []

    for rec in selected:
        if rec["match_status"] != "OK" or rec["vcf"] == "NA":
            per_file.append({
                **rec,
                "parse_status": "MISSING",
                "total_records": "NA",
                "pass_records": "NA",
                "nonpass_records": "NA",
                "recommendation": "MISSING_VCF",
                "recommendation_reason": "No matched raw VCF found.",
            })
            continue

        try:
            stats = parse_vcf(rec["vcf"], max_records=args.max_records_per_vcf)
            recommendation, reason = make_recommendation(stats, include_nonpass=args.include_nonpass)
            total = stats["total_records"]
            pass_n = stats["pass_records"]
            bnd_total = stats["bnd"].get("bnd_total", 0)
            bnd_pass = stats["bnd"].get("bnd_pass", 0)

            row = {
                **rec,
                "parse_status": "OK",
                "total_records": total,
                "pass_records": pass_n,
                "nonpass_records": stats["nonpass_records"],
                "pass_pct": f"{pct(pass_n, total):.4f}",
                "sample_columns": ",".join(stats["sample_names"]),
                "n_sample_columns": len(stats["sample_names"]),
                "n_info_header_ids": len(stats["info_header"]),
                "n_format_header_ids": len(stats["format_header"]),
                "n_filter_header_ids": len(stats["filter_header"]),
                "alt_breakend_records": stats["alt_class_counts"].get("breakend_alt", 0),
                "alt_symbolic_records": stats["alt_class_counts"].get("symbolic_alt", 0),
                "bnd_records_all": bnd_total,
                "bnd_records_pass": bnd_pass,
                "bnd_pct_all": f"{pct(bnd_total, total):.4f}",
                "bnd_pct_pass": f"{pct(bnd_pass, pass_n):.4f}",
                "bnd_alt_mate_parseable": stats["bnd"].get("bnd_alt_mate_parseable", 0),
                "bnd_has_chr2": stats["bnd"].get("bnd_has_chr2", 0),
                "bnd_has_end_or_pos2": stats["bnd"].get("bnd_has_end_or_pos2", 0),
                "bnd_has_mateid": stats["bnd"].get("bnd_has_mateid", 0),
                "bnd_has_strands": stats["bnd"].get("bnd_has_strands", 0),
                "bnd_has_ct": stats["bnd"].get("bnd_has_ct", 0),
                "bnd_interchrom_mate": stats["bnd"].get("bnd_interchrom_mate", 0),
                "bnd_intrachrom_mate": stats["bnd"].get("bnd_intrachrom_mate", 0),
                "bnd_inferable_TRA_by_mate_chr": stats["bnd_infer"].get("bnd_inferable_TRA_by_mate_chr", 0),
                "bnd_has_same_chr_orientation_info": stats["bnd_infer"].get("bnd_has_same_chr_orientation_info", 0),
                "bnd_has_mate_but_no_orientation": stats["bnd_infer"].get("bnd_has_mate_but_no_orientation", 0),
                "bnd_no_reliable_mate": stats["bnd_infer"].get("bnd_no_reliable_mate", 0),
                "info_header_ids": ",".join(stats["info_header"]),
                "format_header_ids": ",".join(stats["format_header"]),
                "filter_header_ids": ",".join(stats["filter_header"]),
                "recommendation": recommendation,
                "recommendation_reason": reason,
            }
            per_file.append(row)

            for scope, counter in [("all", stats["svtype_counts_all"]), ("pass", stats["svtype_counts_pass"])]:
                for sv, n in sorted(counter.items()):
                    svtype_rows.append({**rec, "scope": scope, "svtype": sv, "count": n})
            for flt, n in sorted(stats["filter_counts"].items()):
                filter_rows.append({**rec, "filter": flt, "count": n})
            for key in IMPORTANT_INFO_KEYS:
                info_rows.append({
                    **rec,
                    "info_key": key,
                    "records_with_key": stats["info_presence"].get(key, 0),
                    "records_with_key_pct": f"{pct(stats['info_presence'].get(key, 0), total):.4f}",
                })
            for key, n in sorted(stats["format_presence"].items()):
                format_rows.append({**rec, "format_key": key, "records_with_key": n, "records_with_key_pct": f"{pct(n, total):.4f}"})
            for i, ex in enumerate(stats["first_records"], 1):
                example_rows.append({**rec, "example_no": i, **ex})

        except Exception as exc:
            per_file.append({
                **rec,
                "parse_status": "ERROR",
                "total_records": "NA",
                "pass_records": "NA",
                "nonpass_records": "NA",
                "recommendation": "PARSE_ERROR",
                "recommendation_reason": str(exc),
            })

    per_file_fields = [
        "pair_id", "tumor", "normal", "tool", "match_status", "parse_status", "vcf",
        "match_score", "match_reasons",
        "total_records", "pass_records", "nonpass_records", "pass_pct",
        "sample_columns", "n_sample_columns",
        "n_info_header_ids", "n_format_header_ids", "n_filter_header_ids",
        "alt_breakend_records", "alt_symbolic_records",
        "bnd_records_all", "bnd_records_pass", "bnd_pct_all", "bnd_pct_pass",
        "bnd_alt_mate_parseable", "bnd_has_chr2", "bnd_has_end_or_pos2",
        "bnd_has_mateid", "bnd_has_strands", "bnd_has_ct",
        "bnd_interchrom_mate", "bnd_intrachrom_mate",
        "bnd_inferable_TRA_by_mate_chr", "bnd_has_same_chr_orientation_info",
        "bnd_has_mate_but_no_orientation", "bnd_no_reliable_mate",
        "recommendation", "recommendation_reason",
        "info_header_ids", "format_header_ids", "filter_header_ids",
    ]
    write_tsv(outdir / "raw_vcf_audit.per_file.tsv", per_file, per_file_fields)
    write_tsv(outdir / "raw_vcf_svtype_counts.long.tsv", svtype_rows, ["pair_id", "tumor", "normal", "tool", "vcf", "scope", "svtype", "count"])
    write_tsv(outdir / "raw_vcf_filter_counts.long.tsv", filter_rows, ["pair_id", "tumor", "normal", "tool", "vcf", "filter", "count"])
    write_tsv(outdir / "raw_vcf_info_key_presence.long.tsv", info_rows, ["pair_id", "tumor", "normal", "tool", "vcf", "info_key", "records_with_key", "records_with_key_pct"])
    write_tsv(outdir / "raw_vcf_format_key_presence.long.tsv", format_rows, ["pair_id", "tumor", "normal", "tool", "vcf", "format_key", "records_with_key", "records_with_key_pct"])
    write_tsv(outdir / "raw_vcf_first_records.examples.tsv", example_rows, ["pair_id", "tumor", "normal", "tool", "vcf", "example_no", "chrom", "pos", "id", "alt", "filter", "svtype", "info_keys", "format"])

    # Tool-level summary.
    tool_rows = []
    for tool in tools:
        rows = [r for r in per_file if r.get("tool") == tool and r.get("parse_status") == "OK"]
        rec_n = sum(int(r.get("total_records", 0)) for r in rows)
        pass_n = sum(int(r.get("pass_records", 0)) for r in rows)
        bnd_all = sum(int(r.get("bnd_records_all", 0)) for r in rows)
        bnd_pass = sum(int(r.get("bnd_records_pass", 0)) for r in rows)
        recs = Counter(r.get("recommendation", "NA") for r in rows)
        main_rec = recs.most_common(1)[0][0] if recs else "NO_PARSED_VCF"
        tool_rows.append({
            "tool": tool,
            "parsed_vcf_count": len(rows),
            "total_records": rec_n,
            "pass_records": pass_n,
            "nonpass_records": rec_n - pass_n,
            "pass_pct": f"{pct(pass_n, rec_n):.4f}",
            "bnd_records_all": bnd_all,
            "bnd_records_pass": bnd_pass,
            "bnd_pct_all": f"{pct(bnd_all, rec_n):.4f}",
            "bnd_pct_pass": f"{pct(bnd_pass, pass_n):.4f}",
            "dominant_recommendation": main_rec,
            "recommendation_counts": ";".join(f"{k}:{v}" for k, v in sorted(recs.items())),
        })
    write_tsv(outdir / "raw_vcf_audit.by_tool_summary.tsv", tool_rows, [
        "tool", "parsed_vcf_count", "total_records", "pass_records", "nonpass_records", "pass_pct",
        "bnd_records_all", "bnd_records_pass", "bnd_pct_all", "bnd_pct_pass",
        "dominant_recommendation", "recommendation_counts",
    ])

    readme = outdir / "README_raw_vcf_audit.txt"
    with open(readme, "wt") as handle:
        handle.write(
            "Raw SV VCF audit for BND decision\n"
            "\n"
            "This audit is read-only. It does not create reclassified VCFs.\n"
            "\n"
            "Main files:\n"
            "1. raw_vcf_selected_manifest.tsv: selected raw VCF for each pair x tool.\n"
            "2. raw_vcf_audit.per_file.tsv: record counts, BND field availability, and recommendation per VCF.\n"
            "3. raw_vcf_svtype_counts.long.tsv: SVTYPE count by pair, tool, and scope(all/pass).\n"
            "4. raw_vcf_filter_counts.long.tsv: FILTER distribution.\n"
            "5. raw_vcf_info_key_presence.long.tsv: whether fields such as CHR2, END, STRANDS, CT, MATEID exist.\n"
            "6. raw_vcf_first_records.examples.tsv: first five records from each selected VCF for manual inspection.\n"
            "7. raw_vcf_audit.by_tool_summary.tsv: tool-level summary and recommendation.\n"
            "\n"
            "Interpretation of recommendations:\n"
            "DIRECT_USE_SVTYPE: raw VCF already contains interpretable SVTYPE for most PASS records.\n"
            "NEED_CALLER_SPECIFIC_BND_INFERENCE: do not use generic BND conversion; inspect caller documentation and fields.\n"
            "CAN_INFER_TRA_BUT_NOT_ALL_TYPES: TRA can be inferred from mate chromosomes, but DEL/DUP/INV are not safe.\n"
            "PARTIAL_BND_INFERENCE_ONLY: only a subset of BND records are inferable.\n"
            "KEEP_BND_OR_EXCLUDE_FROM_TYPED_ANALYSIS: BND lacks reliable information for conversion.\n"
        )

    print("===== raw VCF audit done =====")
    print(f"Pairs: {len(pairs)}")
    print(f"Candidate VCFs: {len(candidates)}")
    print(f"Selected rows: {len(selected)}")
    print(f"Output directory: {outdir}")
    print(f"Per-file audit: {outdir / 'raw_vcf_audit.per_file.tsv'}")
    print(f"Tool summary: {outdir / 'raw_vcf_audit.by_tool_summary.tsv'}")
    print(f"README: {readme}")


if __name__ == "__main__":
    main()


遍历输入目录（或文件）
  ↓
对每个 VCF：
  1. 读取 header 和 records
  2. 对每条 record：
     - 不是 BND → 原样保留
     - 是 BND   → 调用 annotate_bnd_info 加注释
  3. 写出新 VCF（header 补 BNDINF 定义，records 逐条写）

# 二、处理步骤

### 运行脚本：

In [ ]:
BASE=/mnt/home/ygjx/chenkejin/share_group_folder_ygjx/Pancreatic_datasets/PDAC_WGS/sv_tools_results

python /mnt/home/ygjx/chenkejin/bnd_format_check/annotate_bnd_with_inferred_svtype.py \
  -i $BASE/cue_result \
     $BASE/delly_result \
     $BASE/lumpy_result \
     $BASE/gridss_result \
     $BASE/svaba_result \
     $BASE/manta_result \
  -o /mnt/home/ygjx/chenkejin/sv_tools_results_bnd_annotated

### 汇总每个工具的转换结果：

In [ ]:
awk -F'\t' '
NR>1{
  count[$1"\t"$4]+=$5
}
END{
  for(k in count) print k"\t"count[k]
}' /mnt/home/ygjx/chenkejin/sv_tools_results_bnd_annotated/bnd_annotated_by_tool.summary.tsv \
| sort